# Dataset Preprocessing

In [ ]:
from pandas import concat, read_csv, to_datetime
from sklearn.model_selection import train_test_split

In [ ]:
dataset = read_csv('../data/raw/collection.csv')
dataset.head()

In [ ]:
dataset = dataset.drop(columns=['Visit', 'Modality', 'Type', 'Format', 'Downloaded'])
dataset = dataset.rename(columns={
    'Image Data ID': 'image_id',
    'Subject': 'subject_id',
    'Group': 'diagnosis',
    'Sex': 'sex',
    'Age': 'age',
    'Description': 'weight',
    'Acq Date': 'date'
})

dataset['image_id'] = dataset['image_id'].str.slice(1)
dataset['date'] = to_datetime(dataset['date']).dt.strftime("%Y-%m-%d")

dataset.to_csv('../data/interim/sampled.csv', header=True, index=False)
dataset.head()

In [ ]:
dataset = dataset[dataset['weight'].str.contains('MPRAGE|MP-RAGE')]
dataset = dataset.drop(columns=['weight'])

dataset = dataset.sort_values(by=['age', 'subject_id'])
oldest_mri = dataset.groupby('subject_id').first().reset_index()
newest_mri = dataset.groupby('subject_id').last().reset_index()
dataset = concat([oldest_mri, newest_mri]).drop_duplicates()

dataset.to_csv('../data/interim/dataset.csv', header=True, index=False)
dataset.head()

In [ ]:
train, remaining = train_test_split(dataset, train_size=0.60, shuffle=True)
valid, test = train_test_split(remaining, train_size=0.50, shuffle=True)

train.to_csv('../data/processed/train.csv', header=True, index=False)
valid.to_csv('../data/processed/valid.csv', header=True, index=False)
test.to_csv('../data/processed/test.csv', header=True, index=False)

print('Total Number of Analysed Subjects:', dataset['subject_id'].nunique())
print('Total Number of Available Images:', dataset['image_id'].nunique())

print('Training Set Size:', len(train))
print('Validation Set Size:', len(valid))
print('Test Set Size:', len(test))

In [ ]:
images_ids = dataset['image_id'].tolist()
print(','.join(images_ids), flush=True)